In [1]:
from pathlib import Path

import duckdb

from thesis.config import load_config
from thesis.vlm.run import configured_clients

ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")
# LOCKED prompt
roleplaying_text = next(p["text"] for p in load_config("prompts")["prompts"] if p["id"] == "roleplaying")

con = duckdb.connect(str(ablation_dir / "manifest.duckdb"), read_only=True)
all_marker_samples = con.sql("SELECT * FROM samples").df()

In [2]:
len(all_marker_samples)

9108

In [7]:
# length of entries inside dir
dir = Path("/Users/phillip/repositories/bachelor-thesis/data/vlm_cache/anthropic")
entries = list(dir.iterdir())

# tissue math, 5036 existing, 1012 of which are red_crisshair+roleplaying, 5 duplicates so 1007
# 9108 - 1007 is 8.101 new entries. Total: 13.137
# (minus a few collisions for the 5 duplicates, since underlying image is dupe independent of marker)
len(entries)

13092

In [4]:
from dotenv import load_dotenv

from thesis.vlm.run import run_dataset

load_dotenv()

ablation_dir = Path("/Users/phillip/repositories/bachelor-thesis/data/ablation")
image_bytes_by_row = [(ablation_dir / str(row.image_path)).read_bytes() for _, row in all_marker_samples.iterrows()]
items = [(image_bytes, roleplaying_text) for image_bytes in image_bytes_by_row]
results = run_dataset(items, clients=configured_clients())